In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode,expr,col,when,concat,hash,concat_ws, array, lpad
from pyspark.sql.types import ShortType, ArrayType, IntegerType

In [2]:
spark = SparkSession.builder.appName("task").getOrCreate()

25/05/22 15:57:43 WARN Utils: Your hostname, milanthapa resolves to a loopback address: 127.0.1.1; using 10.10.42.133 instead (on interface enp2s0)
25/05/22 15:57:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/22 15:57:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

In [ ]:
df_pyspark = spark.read.option('multiline','True').json('/home/milan-thapa/Desktop/Zaki_point_task/files/output/data.json')
df1_pyspark = spark.read.parquet('/home/milan-thapa/Desktop/Zaki_point_task/files/highmark_prv')

In [ ]:
df_pyspark.printSchema()
df1_pyspark.printSchema()


root
 |-- in_network: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- billing_code: string (nullable = true)
 |    |    |-- billing_code_type: string (nullable = true)
 |    |    |-- billing_code_type_version: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- negotiated_rates: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- negotiated_prices: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- additional_information: string (nullable = true)
 |    |    |    |    |    |    |-- billing_class: string (nullable = true)
 |    |    |    |    |    |    |-- billing_code_modifier: array (nullable = true)
 |    |    |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |    |    |-- expiration_date: strin

In [ ]:
np_data = df_pyspark.selectExpr("*","explode (in_network) as net")
net_file = np_data.withColumn("rates",explode("net.negotiated_rates"))
net_file = net_file.withColumn("prices", explode("rates.negotiated_prices"))
net_file = net_file.withColumn('provider',explode("rates.provider_groups"))
net_file = net_file.withColumn('id',explode("provider.npi"))


network_flat = net_file.selectExpr(
        "net.billing_code",
        "net.billing_code_type",
        "net.negotiation_arrangement",
        "prices.billing_class as billing_class",
        "prices.billing_code_modifier as billing_code_modifier",
        "prices.negotiated_rate as negotiated_rate",
        "prices.negotiated_type as negotiated_type",
        "prices.service_code as service_code",
        'id as npi',
        "provider.tin.type as tin_type",
        "provider.tin.value as tin"

    )



In [ ]:
provider_cleaned = network_flat.withColumn('tin', expr("REPLACE(tin, '-', '')"))


In [ ]:
provider_replaced = network_flat.withColumn("tin_type",
                    when(col("tin_type") == "ein", 1)
                    .when(col("tin_type") == "npi", 2))


In [ ]:
df_combine = network_flat.withColumn("provider_group_id",concat("npi", "tin"))


In [ ]:
df_hash = df_combine.withColumn('provider_group_id',hash("provider_group_id"))


In [ ]:
df_new = df_combine.drop('npi', 'tin_type', 'tin')

In [ ]:
df_new.show()

25/05/22 15:44:42 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+--------------------+
|billing_code|billing_code_type|negotiation_arrangement|billing_class|billing_code_modifier|negotiated_rate|negotiated_type|service_code|   provider_group_id|
+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+--------------------+
|         000|           MS-DRG|                    ffs|institutional|                 NULL|       118400.0|        derived|        [21]|126541004723-1476328|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        16358.0|        derived|        [21]|143786595388-3577015|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        33258.5|        derived|        [21]|121592145723-1352166|
|         000|           MS-DRG|              

In [ ]:
df_new.printSchema()  
# Nr of mrf file

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- provider_group_id: string (nullable = true)



In [ ]:
provider_table = df_hash.selectExpr( 'npi',
            "tin_type",
            "tin",
            "provider_group_id")

In [ ]:
provider_table.printSchema()

root
 |-- npi: long (nullable = true)
 |-- tin_type: string (nullable = true)
 |-- tin: string (nullable = true)
 |-- provider_group_id: integer (nullable = false)



In [ ]:
remove_ = provider_table.withColumn('tin', expr("replace(tin,'-','')"))
df_type = remove_.withColumn('tin_type',
        when((col('tin_type')== 'ein'), 1).when((col('tin_type')== 'npi'), 2)
)

In [ ]:
provider_rep = df_type.withColumn("tin_type", col("tin_type").cast(ShortType()))

In [ ]:
provider_rep.show()
# pr of mrf file 

+----------+--------+---------+-----------------+
|       npi|tin_type|      tin|provider_group_id|
+----------+--------+---------+-----------------+
|1265410047|       1|231476328|       -425708255|
|1437865953|       1|883577015|        414463782|
|1215921457|       1|231352166|      -1534245514|
|1962579029|       1|232825878|       -934641354|
|1396850368|       1|231352160|       -603094894|
|1851401756|       1|231352160|       -593143751|
|1356197453|       1|811025608|       1846187399|
|1417971904|       1|200994813|      -1646888026|
|1962401497|       1|251737079|       2088821285|
|1679592380|       1|250969492|        246011291|
|1588269898|       1|250969492|       1376585769|
|1366488108|       1|251260215|       1505474788|
|1578560504|       1|251260215|       -481759795|
|1194744805|       1|250969492|      -1142764743|
|1689679581|       1|251875178|        958137373|
|1285667493|       1|250969492|       -246775706|
|1194744805|       1|250969492|      -1142764743|


In [ ]:
df_dropping = df1_pyspark.drop('prv_fax','provider_name_prefix_text','prv_type_desc')
df_new = df_dropping.withColumn('prv_type_code',
                             when((col('prv_type_code')== 'P'), 1).when((col('prv_type_code')== 'F'), 2)
)

In [ ]:
df_cast = df_new.withColumn("prv_type_code",col("prv_type_code").cast(IntegerType()))
df_merge = df_cast.withColumn("full_name",concat_ws(" ","provider_first_name", "provider_middle_name","provider_last_name"))
df_merge1=df_merge.drop("provider_first_name","provider_last_name","provider_middle_name")


In [ ]:
df_new1 = df_merge1.select(
    "*",  
    col("loc.lat").alias("latitude"),
    col("loc.lon").alias("longitude")
)

In [ ]:
df_new2 = df_new1.drop('loc')

df_array = df_new2.withColumn("taxonomy",array(col("prv_taxonomy_1_code"),col("prv_taxonomy_2_code"),col("prv_taxonomy_3_code")))
df_array1=df_array.drop("prv_taxonomy_1_code","prv_taxonomy_2_code","prv_taxonomy_3_code")


In [ ]:
df_array2 = df_array1.withColumn("prv_specialty",array(col("prv_specialty_1_desc"),col("prv_specialty_2_desc"),col("prv_specialty_3_desc")))
df_array3=df_array2.drop("prv_specialty_1_desc","prv_specialty_2_desc","prv_specialty_3_desc")



In [ ]:
df_array3.show()
df_array3.printSchema()
#new pr table

+----------+--------------------+--------------+---------+-------+--------------------+-------------+----------+--------------------+---------+----------+--------------------+--------------------+
|       npi|        prv_street_1|      prv_city|prv_state|prv_zip|           prv_phone|prv_type_code|       tin|           full_name| latitude| longitude|            taxonomy|       prv_specialty|
+----------+--------------------+--------------+---------+-------+--------------------+-------------+----------+--------------------+---------+----------+--------------------+--------------------+
|1003035353|       501 S 54TH ST|  PHILADELPHIA|       PA|  19143|                NULL|            1|1558759571|       JOEL M. STEIN|39.952625|-75.230324|[2085R0202X, NULL, ]|[Diagnostic Radio...|
|1003035353|       501 S 54TH ST|  PHILADELPHIA|       PA|  19143|                NULL|            1|1821447103|       JOEL M. STEIN|39.952625|-75.230324|[2085R0202X, NULL, ]|[Diagnostic Radio...|
|1013904077|803

In [ ]:
df_join = provider_rep.join(df_array3,how="inner",on=["npi","tin"])
# joining pr table

In [ ]:
df_join.printSchema()

root
 |-- npi: long (nullable = true)
 |-- tin: string (nullable = true)
 |-- tin_type: short (nullable = true)
 |-- provider_group_id: integer (nullable = false)
 |-- prv_street_1: string (nullable = true)
 |-- prv_city: string (nullable = true)
 |-- prv_state: string (nullable = true)
 |-- prv_zip: string (nullable = true)
 |-- prv_phone: string (nullable = true)
 |-- prv_type_code: integer (nullable = true)
 |-- full_name: string (nullable = false)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- taxonomy: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- prv_specialty: array (nullable = false)
 |    |-- element: string (containsNull = true)



In [ ]:
df_bil_code = spark.read.option('header','True').csv('/home/milan-thapa/Desktop/Zaki_point_task/files/billing_taxonomy_list (2).csv')
df_bil_code.printSchema()
# reading csv file

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_description: string (nullable = true)
 |-- taxonomy_list: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)



In [ ]:
df_leftpad = df_bil_code.withColumn("billing_code", lpad(col("billing_code"), 5, "0"))
df_rate2 = df_leftpad.drop('_c4','_c5','_c6')

In [ ]:
df_rate2.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_description: string (nullable = true)
 |-- taxonomy_list: string (nullable = true)



In [ ]:
df_join = df_rate2.select(
    "billing_code",
    "taxonomy_list"
    )

In [ ]:
df_join.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- taxonomy_list: string (nullable = true)



In [ ]:
df_newrtab = df_new.join(df_join,how="inner",on="billing_code")

AnalysisException: [UNRESOLVED_USING_COLUMN_FOR_JOIN] USING column `billing_code` cannot be resolved on the left side of the join. The left-side columns: [`loc`, `npi`, `provider_first_name`, `provider_last_name`, `provider_middle_name`, `prv_city`, `prv_phone`, `prv_specialty_1_desc`, `prv_specialty_2_desc`, `prv_specialty_3_desc`, `prv_state`, `prv_street_1`, `prv_taxonomy_1_code`, `prv_taxonomy_2_code`, `prv_taxonomy_3_code`, `prv_type_code`, `prv_zip`, `tin`].